# Introduction: 

The intention is that this document and the included scripts serve as a template for building a model, 
documenting intermediary steps and 
evaluating the model.

The dataset detail are given in the following sub-sections. 

In [ ]:
knitr::opts_chunk$set(echo=TRUE, warning=TRUE, message=TRUE)

In [ ]:

library("dplyr")
library("mice") #used for inputting missing data
library("ggplot2")
library("scales")
library("ggthemes")
library("randomForest")
library(knitr)
library(corrplot)
library(rpart);library(rpart.plot);  library(partykit)
library(caret)
library(readr) # CSV file I/O, e.g. the read_csv function
library(C50)
library("ROCR")
library(pROC); 
library(plotROC)
library(dplyr)
library(nycflights13)
library(stargazer)
library(pscl)
library(MKmisc); library(ResourceSelection)
library(survey)
library(gbm)
library(GoodmanKruskal)

theme_set(theme_bw())

# date related functions
is.POSIXct  <- function(x) inherits(x, "POSIXct")
is.POSIXlt  <- function(x) inherits(x, "POSIXlt")
is.POSIXt   <- function(x) inherits(x, "POSIXt")
is.Date     <- function(x) inherits(x, "Date")


## Loading the data
The rest of the program assumes a *rectangular, complete and unsummarised* data, with a choice of model type that
describes the model to be built as being constructed to predict the *RESPONSE VARIABLE*
 based on the other (suitable) variables available in the dataset.
We load the data into a single data structure so that we can pre-process in one go. 
The following are the data-related declarations required, primarily `allData` and `responseVariable` 
followed by some `settings'.


In [ ]:

# Data Declarations ---------------------------------------;
 data(churn);
allData             <- rbind(churnTrain, churnTest)  ;	 # DECLARATION POINT ~ RData Frame Object
responseVariable    = "churn" ;			                # DECLARATION POINT ~ Variable "Name"
modelChoice         = 2;                                # DECLARATION POINT ~ 1 for LINEAR, 2 for LOGISTIC

# Settings-------------------------------------------------;
split               = 0.60 ;                    # DECLARATION POINT ~ Dev/Val split
hc                  = 0.75 ;                    # DECLARATION POINT ~ High Correlation Threshold
seedOne             = 123;                      # DECLARATION POINT ~ Seed 1 for sampling replication 
treeDepth           = 3; 

In [ ]:
rawDf <- as.data.frame(allData[complete.cases(allData),], stringsAsFactors = T) # dim(rawDf);

multiplot <- function(..., plotlist=NULL, file, cols=1, layout=NULL) {
              library(grid)
            
              # Make a list from the ... arguments and plotlist
              plots <- c(list(...), plotlist)
            
              numPlots = length(plots)
            
              # If layout is NULL, then use 'cols' to determine layout
              if (is.null(layout)) {
                # Make the panel
                # ncol: Number of columns of plots
                # nrow: Number of rows needed, calculated from # of cols
                layout <- matrix(seq(1, cols * ceiling(numPlots/cols)),
                                ncol = cols, nrow = ceiling(numPlots/cols))
              }
            
             if (numPlots==1) {
                print(plots[[1]])
            
              } else {
                # Set up the page
                grid.newpage()
                pushViewport(viewport(layout = grid.layout(nrow(layout), ncol(layout))))
            
                # Make each plot, in the correct location
                for (i in 1:numPlots) {
                  # Get the i,j matrix positions of the regions that contain this subplot
                  matchidx <- as.data.frame(which(layout == i, arr.ind = TRUE))
            
                  print(plots[[i]], vp = viewport(layout.pos.row = matchidx$row,
                                                  layout.pos.col = matchidx$col))
                }
              }
}

### Data Splitting
A common methodology for model building is to randomly partition the available data into a
training dataset and testing dataset.
Allocation of data to certain tasks (e.g., model development, performance validation) is an important aspect of modeling. 
For this example, the primary interest is to predict the **`r responseVariable`** for future observations, which is not the 
same population as the data used to build the model. This means that, to some degree, we are testing how well the 
model _extrapolates_ to a different population. If we were interested in predicting from the same population 
(i.e., interpolation), taking a simple random sample of the data would be more appropriate. How the training and test sets 
are determined should reflect how the model will be applied.

The development sample for this run will be **`r split*100`%** of all data available. 
The remaining will be used for performance validation.
Function `createDataPartition()` causes balanced splits of the data. If the class argument to this function is a factor (as in this case), 
the random sampling occurs within each class and should preserve the overall class distribution of the data. 
The function requires a random number (set.seed) to be set.

In [ ]:

rawDf[,'responseVariable']  <- as.numeric(as.factor(rawDf[, responseVariable])) - 1     # Converting to `0/1's
rawDf[, responseVariable ]  <- NULL               # Remove the original responseVariable

set.seed(123)
inTraining 	<- createDataPartition(rawDf$responseVariable, p= split, list= FALSE)
devDf 		<- rawDf[ inTraining,];  
valDf 		<- rawDf[-inTraining,];


# Variable manipulations' final stop--------------------------------------;
devDf$fakeNum  <- -1* devDf[,length(devDf)];
valDf$fakeNum  <- -1* valDf[,length(valDf)];

devDf$fakeChar <- LETTERS[round(runif(nrow(devDf), 1, 5),1)]; #str(devDf);
devDf[devDf$responseVariable==1,][1:1000,]$fakeChar <- 'D'
valDf$fakeChar <- LETTERS[round(runif(nrow(valDf), 1, 5),1)]; #str(valDf);
valDf[valDf$responseVariable==1,][1:500,]$fakeChar <- 'D'




set.seed(seedOne)
allVars 		    <- NULL; 
allVars.num         <- NULL; 
allVars.categ       <- NULL;
allVars.char        <- NULL; 
allVars.date        <- NULL; 

devDf.num.select    <- NULL
devDf.categ.select  <- NULL
devDf.char.select   <- NULL
devDf.date.select   <- NULL


all	            <- devDf; all$responseVariable = NULL

allVars 		<- colnames(all);	                            #length(allVars);
allVars.num 	<- colnames(all[sapply(all, is.numeric)]); 	    #length(allVars.num);
allVars.categ 	<- colnames(all[sapply(all, is.factor)]); 	    #length(allVars.categ);
allVars.char    <- colnames(all[sapply(all, is.character)]);    #length(allVars.char); 
allVars.date    <- colnames(all[sapply(all, is.POSIXt)]);       #length(allVars.date);

rm(all);


## Data details

### Raw data

Below is an overview of the data before and after the dev/val split. 
The *1* extra variable in the post-split dataframes is the newly created `responseVariable` that will be used 
in the development exercise.

Object | Data | Observations | Variables
------------- | ------------- | ------------- | -------------
`rawDf` | `r substitute(rawDf)`  | `r dim(rawDf)[1] ` | `r dim(rawDf)[2] `
`devDf` | Development data frame | `r dim(devDf)[1] ` | `r dim(devDf)[2] `
`valDf` | Validation data frame  | `r dim(valDf)[1] ` | `r dim(valDf)[2] `

Below is an overview of the variable *types* in the development sample.

 Variable Class | Type | Count
 ------------- | ------------- | -------------
 **`allVars`**      | **All Variables** | **`r length(allVars) `**
 `allVars.num`  | Numeric       | `r length(allVars.num) `
 `allVars.categ`| Categorical   | `r length(allVars.categ) ` 
 `allVars.char` | Character     | `r length(allVars.char) ` 
 `allVars.date` | Date          | `r length(allVars.date)`
 responseVariable | Factor      | 1


#### Numeric Variables
This is a sorted list

In [ ]:
devDf.num          <- data.frame(devDf[ allVars.num]);     sort(colnames(devDf.num))    # <- allVars.num;    str(devDf.num);     

#### Categorical Variables
This is a sorted list

In [ ]:
devDf.categ        <- data.frame(devDf[ allVars.categ]);   sort(colnames(devDf.categ))  # <- allVars.categ;  str(devDf.categ);   

#### Character Variables
This is a sorted list

In [ ]:
devDf.char         <- data.frame(devDf[ allVars.char]);    sort(colnames(devDf.char))  #    <- allVars.char;   str(devDf.char);    

#### Date Variables
This is a sorted list

In [ ]:
devDf.date         <- data.frame(devDf[ allVars.date]);    sort(colnames(devDf.date))  #  <- allVars.date;   str(devDf.date);    

### Development sample data: Six-point summary
The six-point summary of all variables in given below. 


In [ ]:

summary(devDf)


#### Data Profiling
Data profiling involves creating summary statistics for each and every column and
looking at simple plots of the data to identify trends, clusters or outliers. Summary
statistics can include count, number of missing records, mean / mode / median
values, ranges and quartiles. Box plots are useful tools to visualize some of this
information graphically.
Data profiling helps understand which columns warrant additional attention from
data quality perspective. The appropriate course of action for each column has
to be carefully determined. For some columns, missing values may be replaced by
mean or mode or a constant. Some columns may need to be simply dropped
from analysis.


# Variable Reduction through Correlation and CART (Path 1)

In this section we choose the top 10 variables for model tuning. As a first cut, we use the 
Correlation reduction process for the numeric variables;
and will apply the CART tree to see the splits.


## Correlations between numeric variables
Before we get too deep into the data, let's have a look at what we're dealing with.
Correlations between numeric variables need to checked; and highly 
correlated variables need to be dropped. 'High' correlation threhold is set at `r hc`.



### *All* numeric variables and their correlation coeffs

In [ ]:
descrCorr       <- cor(devDf.num)       #correlation matrix
descrCorr[is.na(descrCorr)] <- 0                            # Missing correlations (NA) are hardcoded as 0
corrplot(descrCorr, method= "color", type= "lower", order= "hclust", tl.cex= 0.75, tl.col= "black", tl.srt= 45, addCoef.col="grey", 
    number.cex = 7/ncol(devDf.num), number.digits=2)


highCorr            <- findCorrelation(descrCorr, hc)   #index of highly correlated vars;
devDf.num.select    <- data.frame(devDf.num[, -highCorr]);      

# Variables that are dropped
# colnames(devDf.num[, highCorr])
# colnames(devDf.num.select) 
# ncol(devDf.num.select)


### *Select* variables after adjusting for correlation.

In [ ]:

descrCorr       <- cor(devDf.num.select) #correlation matrix
descrCorr[is.na(descrCorr)] <- 0
corrplot(descrCorr, method = "color", type="lower", order="hclust", tl.cex = 0.75, tl.srt = 45, tl.col= "black", addCoef.col="grey",
    number.cex= 7/ncol(devDf.num.select), number.digits=2)


## Visualising the select numeric variables
Listing of the numeric variables  `devDf.num.select` starting with `responseVariable` as the ideal distribution.

In [ ]:
p1fn <- function(fx) {	
    	ggplot(Df1, aes(x= fx, y= responseVariable)) + 
	        geom_point(alpha= 1/33) + xlab(names(fx)) + 
	        geom_smooth() +
	        labs(title='Population Density and Default Trend')
}

p2fn <- function(fx) {
        p1 <- ggplot(Df1, aes_string(colnames(round(fx,2)), fill=as.factor(Df1$responseVariable))) + 
                geom_density(alpha = 0.3)   + 
                xlab(colnames(fx)) +
    		    scale_fill_discrete(name="Response\nVariable") +
    		    labs(title='Population Density and Default Trend')
    		    
        p2 <- ggplot(Df1, aes_string(colnames(round(fx,2)), y= 'responseVariable')) + 
    		    xlab(colnames(fx)) +
    		    geom_point() + 
    		    stat_summary(fun.y= "mean", colour= "red",  geom= "point") + 
    		    geom_count(aes(color= ..prop.., group= 1)) 
        
    	multiplot(p1, p2, cols= 1)
}

Df1     	<- devDf[,c( 'responseVariable', colnames(devDf.num.select) )] 

ifelse(modelChoice == 1,
        ( lapply(seq_along(Df1), function(x) {p1fn(Df1[x])}) ) 
    ,ifelse(modelChoice == 2,
        ( lapply(seq_along(Df1), function(x) {p2fn(Df1[x])}) )
        ,
    )
)

## Visualising the character variables
Listing of the character (and category) variables:

In [ ]:
## Bar Charts for all character variables~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
#for (i in 1:length(colnames(devDf.char)) ){ 
#	x = devDf[, allVars.char[i]]
#	#print(aggregate(caretDf[, 'responseVariable'], list(x),  mean)) # Edit for maximum display
#	#print(ggplot(devDf, aes_string(as.name(colnames(devDf.char[i])),'responseVariable')) 
#	    + geom_bar(stat = "summary", fun.y = "mean") + theme_bw())
#	print(ggplot(devDf, aes(as.name(colnames(devDf.char[i])),responseVariable)) + geom_bar(stat = "summary", fun.y = "mean") + theme_bw())
#}

# or

str(devDf.char);
colnames(devDf.char);
colnames(devDf.categ);

Df     	<- devDf[,c( 'responseVariable', colnames(devDf.char), colnames(devDf.categ) )] 

str(Df)
#lapply(Df,  function(fx) {
#    (ggplot(Df, aes(x= reorder(fx, -responseVariable),y= responseVariable)) + geom_bar(stat= "summary", fun.y= "mean") 
#	    + theme_bw() + ylab("Mean of responseVariable")  )
#})

p1fc <- function(fx) {	

    	ggplot(Df, aes(x= reorder(fx, -responseVariable), y= responseVariable)) + 
	        geom_bar(stat= "summary", fun.y= "mean" ) + 
	        xlab(nm)  +
    	    labs(title='Default Rate by Category')
}

( lapply(seq_along(Df), function(x) { nm <<- colnames(Df[x]); p1fc(Df[,x]) }) ) 





### CART Tree

Tree models are computationally intensive techniques for recursively partitioning response variables into subsets based on 
their relationship to one or more (usually many) predictor variables.  Classification trees involve a categorical response 
variable and regression trees a continuous response variable; the approaches are similar enough to be referred to together 
as CART (Classification and Regression Trees) [http://plantecology.syr.edu/fridley/bio793/cart.html].  
Predictors can be any combination of categorical or continuous variables.
Apply the CART tree to see the splits by character variables.

In [ ]:
caretDf     <- devDf[, c('responseVariable', allVars.char) ]
form        <- formula(paste('responseVariable', "~ ."))
ctrl        <- rpart.control(maxdepth= 3)
treeSplit   <- rpart(form, caretDf, control=ctrl)
#print(treeSplit)
rpart.plot(treeSplit)
tmp         <- printcp(treeSplit) #print(1-tmp[,c(3,4)]  )


Apply the CART tree to see the splits by *all*  variables

In [ ]:
caretDf     <- devDf[, c('responseVariable', allVars) ]
form        <- formula(paste('responseVariable', "~ ."))
ctrl        <- rpart.control(maxdepth= 3)
treeSplit   <- rpart(form, caretDf, control=ctrl)
as.party(treeSplit)
#print(treeSplit)
rpart.plot(treeSplit)
tmp         <- printcp(treeSplit) #print(1-tmp[,c(3,4)]  )


Apply the CART tree to see the splits by *devDf.num.select*  variables

In [ ]:
caretDf     <- devDf[, c('responseVariable', colnames(devDf.num.select)) ]
form        <- formula(paste('responseVariable', "~ ."))
ctrl        <- rpart.control(maxdepth= 3)
treeSplit   <- rpart(form, caretDf, control=ctrl)
as.party(treeSplit)
#print(treeSplit)
rpart.plot(treeSplit)
tmp         <- printcp(treeSplit) #print(1-tmp[,c(3,4)]  )

allVars.select.tree <- names(treeSplit$variable.importance)
allVars.select.tree.10 <- names(treeSplit$variable.importance[1:10])


### Selected variables - Tree Top 10
The variables listed in `allVars.select.tree.10` are the top 10 variables selected for the model.

In [ ]:
allVars.select <- allVars.select.tree.10;
allVars.select;

#### Numeric
The **`\r ncol(devDf.num.select)`** variables that are selected are stored in the vector `allVars.num.select`. \Trick.

In [ ]:
allVars.num.select <- colnames(devDf.num.select);
allVars.num.select;

#### Categorical
The **`\r ncol(devDf.categ.select)`** variables that are selected are stored in the vector `allVars.categ.select`.

In [ ]:
allVars.categ.select <- colnames(devDf.categ.select)
allVars.categ.select;

#### Character
The **`\r ncol(devDf.char.select)`** variables that are selected are stored in the vector `allVars.char.select`.

In [ ]:
allVars.char.select <- colnames(devDf.char.select)
allVars.char.select;


#### Date
The **`\r ncol(devDf.date.select)`** variables that are selected are stored in the vector `allVars.date.select`.

In [ ]:
allVars.date.select <- colnames(devDf.date.select)
allVars.date.select;
allVars.select <- c(allVars.num.select, allVars.categ.select, allVars.char.select, allVars.date.select)


### Dropped variables

#### Numeric
The **`r ncol(devDf.num[, highCorr])`** variables that are dropped are stored in the vector `devDf.num.select[, highCorr]` 

In [ ]:
colnames(devDf.num[, highCorr])

#### Categorical
The **`\r ncol(devDf.categ[, highCorr])`** variables that are dropped are stored in the vector `devDf.categ.select[, highCorr]` 

In [ ]:
#colnames(devDf.categ[, highCorr])

#### Character
The **`\r ncol(devDf.categ[, highCorr])`** variables that are dropped are stored in the vector `devDf.char.select[, highCorr]` 

In [ ]:
#colnames(devDf.char[, highCorr])

## Logistic Model Coeffs with Tree Variables
Presented below is the `r ifelse(modelChoice==1, "linear", "logistic") ` regression model with all the variables listed under sections `Selected variables` 
stored under `allVars.select`. This is the final list.

In [ ]:

 caretDf     <- devDf[, c('responseVariable', allVars.select) ]
#caretDf     <- devDf[, c('responseVariable', allVars.num.select) ]


# Model Training
# control parameters
lmControl <- trainControl(method= "repeatedcv" 
                        ,number= 4                    
                        ,repeats= 2          
)

glmControl <- trainControl(method = "repeatedcv" 
                        ,repeats = 5
)

set.seed(825)

ifelse(     modelChoice == 1,
        lmFit   <- train(responseVariable ~ . 
                    ,data= caretDf
                    ,method= "lm" 
                    ,trControl= lmControl
                    ,metric= "Rsquared"
    )
    ,ifelse(modelChoice == 2,
        lmFit   <- train(responseVariable ~ . 
                    ,data= caretDf
                    ,method= "glm" ,family= "binomial"
                    ,trControl= glmControl
        )
    ,)
)


ifelse(modelChoice == 1,
    lmFit <- lm(responseVariable ~ . , data= caretDf
    ) 
    ,ifelse(modelChoice == 2,
        lmFit <- glm(responseVariable ~ . , data= caretDf, family = "binomial"
        ) 
        ,
    )
)
 


# Variable Reduction through Stepwise (Path 2)


## Stepwise Process
Stepwise Logistic Regression with Akaike information criterion (AIC = 2k - 2 log L) as the choice metric.

In [ ]:
 fullMod = glm(responseVariable ~ ., data= devDf, family= binomial)
 summary(fullMod)
 intMod <- glm(responseVariable ~ 1, data= devDf, family= binomial)
 summary(intMod)
 fwdSelection = step(intMod, scope=list(lower=formula(intMod),upper=formula(fullMod)), direction="forward")
 formula(fwdSelection)
 summary(fwdSelection)
 
 

# Model Coefficients
Coefficients of independent variables in the of the 
tree-based model `lmFit` and 
stepwise-based model `fwdSelection`.

Signif. codes:  0 '\*\*\*' 0.001 '\*\*' 0.01 '*' 0.05 '.' 0.1 ' ' 1


In [ ]:

stargazer(  lmFit, fwdSelection
            ,type= 'html'
            ,intercept.bottom = FALSE
            ,digits = 4
            ,column.labels = c("Tree-based", "Stepwise")
)


# Model Testing [Prediction] *(on maximum 40000 obs)*
## Tree-based model

The generated model coefficients are tested on the validation data. 
*In this run, they're run on the maximum of 40000. This condition should be removed in the final runs.*  

In [ ]:

test.valDf      <- valDf[1:(min(40000, nrow(valDf))),];

#ifelse(modelChoice == 1,
#        test.valDf[,'predictionVariable']   <- predict(lmFit, newdata = test.valDf, type='response')
#    ,ifelse(modelChoice == 2,
#            test.valDf[,'predictionVariable']   <- predict(lmFit, newdata = test.valDf, type='response')
#    ,
#    )
#)
# section above can go.... repeats

ifelse(modelChoice == 1,
        test.valDf[,'predictionVariable']   <-  round(predict(lmFit, newdata = test.valDf, type='response'),2)
    ,ifelse(modelChoice == 2,
            test.valDf[,'predictionVariable']    <- round(predict(lmFit, newdata = test.valDf, type='response'),2)
    ,
    )
)

str(test.valDf)


### Classification Rate (assuming cut-off of **0.70**), ANOVA, and ROC
When developing models for prediction, the most critical metric regards how well the model does in predicting the 
target variable on out of sample observations. The process involves using the model estimates to predict values on the training set. 
Afterwards, we will compared the predicted target variable versus the observed values for each observation. 

The difference between the null deviance and the residual deviance shows how our model is doing against the null model (a model with only the intercept). The wider this gap, the better. Analyzing the table we can see the drop in deviance when adding each variable one at a tim

While no exact equivalent to the R2 of linear regression exists, the McFadden R2 index can be used to assess the model fit.

In [ ]:
ifelse(modelChoice == 2,
        mc2 <- list(
                    confusionMatrix((test.valDf$predictionVariable > .70), (test.valDf$responseVariable == 1))
                    ,anova(lmFit, test="Chisq")
                    ,pR2(lmFit)
                    )
        ,mc2 <- list(
                    anova(lmFit, test="Chisq")
                    )
)

In [ ]:
print(mc2)


We can see that the Accuracy of the classification is `r mc2[[1]]$overall['Accuracy']`. However, the Kappa being 
generally better statisitic to look at, is `r mc2[[1]]$overall['Kappa']`.


### Comparison of Actual v/s Predicted 

`r ifelse(modelChoice == 2, print("Since modelChoice was logistic we can generate AUC, KS and other lift statistics to assess the performance."), print("Since valDf$responseVariable is a factor we can generate a confusion matrix to assess the accuracy and Kappa statistics."))`

In [ ]:

ifelse (modelChoice == 2, 
        {
            pr  <- prediction(test.valDf$predictionVariable, test.valDf$responseVariable)
            auc <- performance(pr, measure= "auc")@y.values[[1]]
            Df1  <- test.valDf[, c('responseVariable', 'predictionVariable')]
            ks  <- ks.test(Df1$predictionVariable[Df1$responseVariable == 1], Df1$predictionVariable[Df1$responseVariable == 0])
            a   <- list(
                        
                        paste("AUC ->", round(auc,4))
                        ,lapply(seq_along(Df1), function(x) {p2fn(Df1[x])})  
                        ,ggplot(Df1, aes(x= predictionVariable, colour=as.factor(responseVariable) )) + stat_ecdf() + annotate("text", label= paste("Max KS=",round(ks$statistic,2)), x = 0.75* max(Df1$predictionVariable), y= max(Df1$predictionVariable))  + labs(colour = "Response\nVariable")
                        )
        } 
        ,ifelse(modelChoice == 1,
                a   <- paste("R-squared ->",round(summary(lmFit)$r.squared,4), "Adj.R-squared ->", round(summary(lmFit)$adj.r.squared,4))
        ,
        )    
)
a; rm(a);


#Df     	<- test.valDf[, c('responseVariable', 'predictionVariable')] 
#for (i in 2:length(Df)) {
#    p1 <- ggplot(Df, aes(x=round((Df)[,i],2), fill=as.factor(responseVariable))) + geom_density(alpha = 0.3) + xlab(colnames(Df)[i])
#    p2 <- ggplot(Df, aes(round((Df)[,i],2), y = responseVariable)) + 
#		xlab(colnames(Df)[i]) +
#	    geom_point() + 
#		stat_summary(fun.y= "mean", colour= "red",  geom= "point") + 
#		geom_count(aes(color= ..prop.., group= 1)) 
#   multiplot(p1, p2, cols=1)
#}


## Stepwise model

The generated model coefficients are tested on the validation data. 
*In this run, they're run on the maximum of 40000. This condition should be removed in the final runs.*  

In [ ]:

test.valDf      <- valDf[1:(min(40000, nrow(valDf))),];

#ifelse(modelChoice == 1,
#        test.valDf[,'predictionVariable']   <- predict(fwdSelection, newdata = test.valDf, type='response')
#    ,ifelse(modelChoice == 2,
#            test.valDf[,'predictionVariable']   <- predict(fwdSelection, newdata = test.valDf, type='response')
#    ,
#    )
#)
# section above can go.... repeats

ifelse(modelChoice == 1,
        test.valDf[,'predictionVariable']   <-  round(predict(fwdSelection, newdata = test.valDf, type='response'),2)
    ,ifelse(modelChoice == 2,
            test.valDf[,'predictionVariable']    <- round(predict(fwdSelection, newdata = test.valDf, type='response'),2)
    ,
    )
)

str(test.valDf)


### Classification Rate (assuming cut-off of **0.70**), ANOVA, and ROC
When developing models for prediction, the most critical metric regards how well the model does in predicting the 
target variable on out of sample observations. The process involves using the model estimates to predict values on the training set. 
Afterwards, we will compared the predicted target variable versus the observed values for each observation. 

The difference between the null deviance and the residual deviance shows how our model is doing against the null model (a model with only the intercept). The wider this gap, the better. Analyzing the table we can see the drop in deviance when adding each variable one at a tim

While no exact equivalent to the R2 of linear regression exists, the McFadden R2 index can be used to assess the model fit.

In [ ]:
ifelse(modelChoice == 2,
        mc2 <- list(
                    confusionMatrix((test.valDf$predictionVariable > .70), (test.valDf$responseVariable == 1))
                    ,anova(fwdSelection, test="Chisq")
                    ,pR2(fwdSelection)
                    )
        ,mc2 <- list(
                    anova(fwdSelection, test="Chisq")
                    )
)

In [ ]:
print(mc2)




We can see that the Accuracy of the classification is `r mc2[[1]]$overall['Accuracy'] `. However, the Kappa being 
generally better statisitic to look at, is `r mc2[[1]]$overall['Kappa']`.


### Comparison of Actual v/s Predicted 

`r ifelse(modelChoice == 2, print("Since modelChoice was logistic we can generate AUC, KS and other lift statistics to assess the performance."), print("Since valDf$responseVariable is a factor we can generate a confusion matrix to assess the accuracy and Kappa statistics."))`

In [ ]:

ifelse (modelChoice == 2, 
        {
            pr  <- prediction(test.valDf$predictionVariable, test.valDf$responseVariable)
            auc <- performance(pr, measure= "auc")@y.values[[1]]
            Df1  <- test.valDf[, c('responseVariable', 'predictionVariable')]
            ks  <- ks.test(Df1$predictionVariable[Df1$responseVariable == 1], Df1$predictionVariable[Df1$responseVariable == 0])
            a   <- list(
                        
                        paste("AUC ->", round(auc,4))
                        ,lapply(seq_along(Df1), function(x) {p2fn(Df1[x])})  
                        ,ggplot(Df1, aes(x= predictionVariable, colour=as.factor(responseVariable) )) + stat_ecdf() + annotate("text", label= paste("Max KS=",round(ks$statistic,2)), x = 0.75* max(Df1$predictionVariable), y= max(Df1$predictionVariable)) + labs(colour = "Response\nVariable")
                        )
        } 
        ,ifelse(modelChoice == 1,
                a   <- paste("R-squared ->",round(summary(lmFit)$r.squared,4), "Adj.R-squared ->", round(summary(lmFit)$adj.r.squared,4))
        ,
        )    
)
a; rm(a);


#Df     	<- test.valDf[, c('responseVariable', 'predictionVariable')] 
#for (i in 2:length(Df)) {
#    p1 <- ggplot(Df, aes(x=round((Df)[,i],2), fill=as.factor(responseVariable))) + geom_density(alpha = 0.3) + xlab(colnames(Df)[i])
#    p2 <- ggplot(Df, aes(round((Df)[,i],2), y = responseVariable)) + 
#		xlab(colnames(Df)[i]) +
#	    geom_point() + 
#		stat_summary(fun.y= "mean", colour= "red",  geom= "point") + 
#		geom_count(aes(color= ..prop.., group= 1)) 
#   multiplot(p1, p2, cols=1)
#}


# Conclusion

In Summary, this report is a **start** of an approach which is now being adopted by industry to expedite
the common, non-decisive tasks.

Thank you for taking the time to read my report. Please get in touch if you have any questions.
Special thanks to the `R` Core Team, Kaggle, and developers of `R` packages, especially `caret`.
I would like to credit [Megan Ridsal](https://www.kaggle.com/mrisdal/titanic/exploring-survival-on-the-titanic) 
a great tutorial that i've taken great inspiration from.
Special thanks to the numerous references on R and the packages.
1. https://cran.r-project.org/web/packages/GoodmanKruskal/vignettes/GoodmanKruskal.html 
2. Respective creators of all the libraries listed in Section 1.








